# Bài 59: Ước lượng tư thế camera một lượt xuôi (Feed-Forward Camera Pose Estimation)

RANSAC ở Bài 23 và cấu trúc từ chuyển động ở Bài 29 khôi phục hình học camera thông qua một quy trình lặp: phát hiện đặc trưng, so khớp chúng, chạy RANSAC để loại bỏ ngoại lai, giải ma trận thiết yếu, phân rã nó thành phép xoay và tịnh tiến. Mỗi giai đoạn là một thuật toán riêng biệt, thiết kế thủ công. **VGGT** (Visual Geometry Grounded Transformer, <a href="../references.html#wang-2024-vggt">Wang et al., 2025</a>) thay toàn bộ quy trình bằng một mạng một lượt xuôi (feed-forward) duy nhất: đưa vào các cặp tương ứng (hoặc thậm chí ảnh thô) từ nhiều góc nhìn, và nó hồi quy trực tiếp tư thế camera trong một lượt xuôi, không có bước RANSAC tường minh nào ở bất cứ đâu. Bài này xây dựng một phiên bản nhỏ của ý tưởng đó và so sánh nó với quy trình cổ điển mà nó thay thế &mdash; bao gồm chính xác nơi mỗi cách thắng.

In [ ]:
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Các cặp tương ứng hai góc nhìn tổng hợp, có ngoại lai (Synthetic two-view correspondences, with outliers)
Một đám mây điểm 3D ngẫu nhiên, được chiếu vào hai góc nhìn camera liên hệ bởi một phép xoay ngẫu nhiên đã biết và một *hướng* tịnh tiến (phép chiếu lỗ kim ở Bài 26). *Hướng* tịnh tiến là nhiều nhất mà một cặp hai góc nhìn đã hiệu chỉnh có thể khôi phục &mdash; tỉ lệ tuyệt đối về bản chất không quan sát được chỉ từ hai góc nhìn (cùng sự mơ hồ tỉ lệ đơn mắt ở Bài 56, dưới một dạng khác). Một tỉ lệ có thể điều khiển các cặp tương ứng được thay bằng các cặp khớp nhầm ngẫu nhiên, mô phỏng các cặp khớp sai mà RANSAC ở Bài 23 được xây dựng để chịu đựng.

In [ ]:
K = np.array([[50., 0, 32], [0, 50., 32], [0, 0, 1]])
N_POINTS = 40

def random_rotation(rng, max_angle_deg=30):
    axis = rng.normal(size=3); axis /= np.linalg.norm(axis)
    angle = np.radians(rng.uniform(-max_angle_deg, max_angle_deg))
    Kx = np.array([[0, -axis[2], axis[1]], [axis[2], 0, -axis[0]], [-axis[1], axis[0], 0]])
    return np.eye(3) + np.sin(angle) * Kx + (1 - np.cos(angle)) * (Kx @ Kx)

def make_pair(rng, n_points=N_POINTS, outlier_frac=0.0):
    pts3d = rng.uniform(-2, 2, (n_points, 3)) + np.array([0, 0, 8])
    R_true = random_rotation(rng)
    t_dir = rng.uniform(-1, 1, 3); t_dir /= np.linalg.norm(t_dir)

    def project(pts, R, t):
        cam_pts = (R @ pts.T).T + t
        proj = (K @ cam_pts.T).T
        return proj[:, :2] / proj[:, 2:3]

    pts1 = project(pts3d, np.eye(3), np.zeros(3))
    pts2 = project(pts3d, R_true, t_dir * 2.0)

    n_outliers = int(outlier_frac * n_points)
    if n_outliers > 0:
        idx = rng.choice(n_points, n_outliers, replace=False)
        pts2[idx] = rng.uniform(0, 64, (n_outliers, 2))

    K_inv = np.linalg.inv(K)  # normalize to camera coordinates, so the network never needs to learn K
    npts1 = (K_inv @ np.hstack([pts1, np.ones((n_points, 1))]).T).T[:, :2]
    npts2 = (K_inv @ np.hstack([pts2, np.ones((n_points, 1))]).T).T[:, :2]
    return npts1.astype(np.float32), npts2.astype(np.float32), R_true.astype(np.float32), t_dir.astype(np.float32)

def rotation_angle_error(R_est, R_true):
    R_diff = R_est.T @ R_true
    cos_angle = np.clip((np.trace(R_diff) - 1) / 2, -1, 1)
    return np.degrees(np.arccos(cos_angle))

def translation_angle_error(t_est, t_true):
    t_est = t_est / (np.linalg.norm(t_est) + 1e-8)
    t_true = t_true / np.linalg.norm(t_true)
    return np.degrees(np.arccos(np.clip(np.dot(t_est, t_true), -1, 1)))

## Quy trình cổ điển: ma trận thiết yếu + RANSAC (The classical pipeline: essential matrix + RANSAC)
`cv2.findEssentialMat` với `RANSAC` (Bài 23) theo sau là `cv2.recoverPose` (Bài 28) &mdash; cách tiếp cận tiêu chuẩn trong sách giáo khoa cho tư thế tương đối hai góc nhìn, và chính xác là quy trình mà các mạng kiểu VGGT được xây dựng để thay thế.

In [ ]:
def classical_pose(npts1, npts2):
    pts1_px = (K @ np.hstack([npts1, np.ones((N_POINTS, 1))]).T).T[:, :2].astype(np.float32)
    pts2_px = (K @ np.hstack([npts2, np.ones((N_POINTS, 1))]).T).T[:, :2].astype(np.float32)
    E, mask = cv2.findEssentialMat(pts1_px, pts2_px, K, method=cv2.RANSAC, prob=0.999, threshold=1.0)
    if E is None:
        return np.eye(3), np.array([0, 0, 1.0])
    _, R, t, _ = cv2.recoverPose(E, pts1_px, pts2_px, K)
    return R, t.ravel()

rng = np.random.default_rng(0)
for outlier_frac in [0.0, 0.2, 0.4]:
    rot_errs, trans_errs = [], []
    for _ in range(20):
        p1, p2, R_true, t_true = make_pair(rng, outlier_frac=outlier_frac)
        R_est, t_est = classical_pose(p1, p2)
        rot_errs.append(rotation_angle_error(R_est, R_true))
        trans_errs.append(translation_angle_error(t_est, t_true))
    print(f'outlier_frac={outlier_frac}: classical rot err={np.mean(rot_errs):.2f} deg, '
          f'trans dir err={np.mean(trans_errs):.2f} deg')

## Một mạng tư thế một lượt xuôi (A feed-forward pose network)
Mỗi cặp tương ứng `(x1, y1, x2, y2)` trở thành một token; một bộ mã hóa Transformer nhỏ (Bài 46) cho phép mọi cặp tương ứng chú ý tới mọi cặp khác — đây là thành phần mà một kiến trúc gộp cực đại kiểu PointNet ngây thơ còn thiếu: khôi phục một mối quan hệ hình học hai góc nhìn (đặc biệt là hướng tịnh tiến) về bản chất đòi hỏi suy luận về cách các cặp tương ứng liên hệ *với nhau*, không chỉ xử lý mỗi cặp một cách độc lập rồi gộp lại. Phép xoay được hồi quy qua một biểu diễn 6D liên tục (<a href="../references.html#zhou-2019-rotation6d">Zhou et al., 2019</a> — hai vector 3D, được trực chuẩn hóa Gram-Schmidt thành một ma trận xoay hợp lệ, tránh các điểm gián đoạn của quaternion hay góc Euler), và hướng tịnh tiến dưới dạng một vector 3 chiều đơn vị.

In [ ]:
def rot6d_to_matrix(x):
    a1, a2 = x[..., :3], x[..., 3:]
    b1 = F.normalize(a1, dim=-1)
    b2 = a2 - (b1 * a2).sum(-1, keepdim=True) * b1
    b2 = F.normalize(b2, dim=-1)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

class PoseNet(nn.Module):
    def __init__(self, hidden=64, n_heads=4, n_layers=2):
        super().__init__()
        self.embed = nn.Linear(4, hidden)
        layer = nn.TransformerEncoderLayer(hidden, n_heads, dim_feedforward=hidden * 2,
                                            batch_first=True, dropout=0.0)
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 9))

    def forward(self, corr):  # corr: (B, N, 4) = [x1, y1, x2, y2]
        tok = self.encoder(self.embed(corr))
        pooled = tok.mean(dim=1)
        out = self.head(pooled)
        R = rot6d_to_matrix(out[..., :6])
        t = F.normalize(out[..., 6:], dim=-1)
        return R, t

def make_batch(rng, batch_size, outlier_frac):
    corrs, Rs, ts = [], [], []
    for _ in range(batch_size):
        p1, p2, R, t = make_pair(rng, outlier_frac=outlier_frac)
        corrs.append(np.concatenate([p1, p2], axis=1))
        Rs.append(R); ts.append(t)
    return (torch.tensor(np.array(corrs)), torch.tensor(np.array(Rs)), torch.tensor(np.array(ts)))

torch.manual_seed(0)
model = PoseNet()
opt = torch.optim.Adam(model.parameters(), lr=0.001)
train_rng = np.random.default_rng(500)
for epoch in range(1400):
    outlier_frac = train_rng.uniform(0, 0.5)  # train across a range of outlier ratios
    corr, R_true, t_true = make_batch(train_rng, 32, outlier_frac)
    R_pred, t_pred = model(corr)
    rot_loss = ((R_pred - R_true) ** 2).sum(dim=(1, 2)).mean()
    trans_loss = (1 - (t_pred * t_true).sum(-1)).mean()
    loss = rot_loss + trans_loss
    opt.zero_grad()
    loss.backward()
    opt.step()

print(f'final training loss: {loss.item():.4f}')

## Đối đầu trực tiếp: RANSAC cổ điển và mạng một lượt xuôi, qua các tỉ lệ ngoại lai (Head to head: classical RANSAC vs. feed-forward network, across outlier ratios)

In [ ]:
eval_rng = np.random.default_rng(999)
results = {}
for outlier_frac in [0.0, 0.2, 0.4]:
    learned_rot, learned_trans, classical_rot, classical_trans = [], [], [], []
    for _ in range(20):
        p1, p2, R_true, t_true = make_pair(eval_rng, outlier_frac=outlier_frac)
        with torch.no_grad():
            corr = torch.tensor(np.concatenate([p1, p2], axis=1))[None]
            R_pred, t_pred = model(corr)
        learned_rot.append(rotation_angle_error(R_pred[0].numpy(), R_true))
        learned_trans.append(translation_angle_error(t_pred[0].numpy(), t_true))

        R_c, t_c = classical_pose(p1, p2)
        classical_rot.append(rotation_angle_error(R_c, R_true))
        classical_trans.append(translation_angle_error(t_c, t_true))

    results[outlier_frac] = (np.mean(learned_rot), np.mean(learned_trans), np.mean(classical_rot), np.mean(classical_trans))
    print(f'outlier_frac={outlier_frac}:')
    print(f'  learned:   rot err={np.mean(learned_rot):.2f} deg, trans dir err={np.mean(learned_trans):.2f} deg')
    print(f'  classical: rot err={np.mean(classical_rot):.2f} deg, trans dir err={np.mean(classical_trans):.2f} deg')

## Nơi mỗi cách tiếp cận thắng (Where each approach wins)
Trên dữ liệu sạch (không ngoại lai), RANSAC cổ điển thắng rõ ràng — một phép giải hình học chính xác, có điều kiện tốt rất khó bị đánh bại khi đầu vào thực sự thỏa mãn các giả định của nó. Khi tỉ lệ ngoại lai tăng, khoảng cách thu hẹp rồi **đảo ngược**: mạng một lượt xuôi, được huấn luyện trên cả một khoảng tỉ lệ ngoại lai, trở nên *bền vững hơn* RANSAC với ngưỡng và ngân sách vòng lặp cố định. Đây không phải là sự tình cờ của thiết lập đồ chơi này — đó là động lực thực sự, đã được ghi nhận cho các kiến trúc kiểu VGGT: tính bền vững của RANSAC bị giới hạn bởi các siêu tham số của nó và bởi mô hình hình học mà nó giả định (một ma trận thiết yếu cứng duy nhất liên hệ chính xác hai góc nhìn), trong khi một mạng được huấn luyện trên đủ dữ liệu tương ứng đa dạng, lộn xộn, thực tế học được thứ gì đó gần với "một tư thế hợp lý trông như thế nào với bằng chứng như thế này", và suy giảm một cách êm thấm hơn.

Điều này phản chiếu gần như chính xác câu chuyện ảnh nổi ở Bài 58: một thuật toán cố định, thiết kế thủ công (so khớp khối, RANSAC) chính xác dưới các giả định lý tưởng của nó và mong manh bên ngoài chúng; một mạng được huấn luyện trên một phân phối các điều kiện đánh đổi một chút độ chính xác đỉnh đó lấy tính bền vững tốt hơn nhiều trên toàn bộ phạm vi các điều kiện thực mà nó thực sự sẽ gặp. Không cái nào thay thế hoàn toàn cái kia — các quy trình thị giác 3D trong sản xuất ngày càng dùng chính loại mạng một lượt xuôi này như một bước *khởi tạo* nhanh, bền vững, với một bước tinh chỉnh hình học cổ điển (tinh chỉnh bó, Bài 29) như bước dọn dẹp cuối cùng với độ chính xác tuyệt đối khi tài nguyên tính toán cho phép.

### Bài tập (Exercises)
1. Tăng `max_angle_deg` trong `random_rotation` từ 30 lên 90. Khoảng cách giữa hiệu năng học được và cổ điển có thay đổi không, và sự ưa thích đã biết của RANSAC cổ điển đối với các đường cơ sở nhỏ, có điều kiện tốt có giải thích được chiều của sự thay đổi không?
2. Loại bỏ phép tự chú ý (`self.encoder(self.embed(corr))` → chỉ `self.embed(corr)`, rồi gộp trung bình trực tiếp) và huấn luyện lại. Việc loại bỏ chú ý giữa các cặp tương ứng có gây hại cho độ chính xác phép xoay, độ chính xác tịnh tiến, hay cả hai — và điều đó có khớp với khẳng định rằng riêng hướng tịnh tiến cần suy luận *quan hệ* giữa các điểm không?
3. Huấn luyện hai mô hình riêng biệt, một mô hình chỉ luôn thấy `outlier_frac=0.0` trong khi huấn luyện và một mô hình chỉ luôn thấy `outlier_frac=0.5`, rồi đánh giá cả hai trên toàn bộ khoảng `[0.0, 0.5]`. Có mô hình chuyên biệt nào vượt mô hình đa năng (được huấn luyện trên toàn bộ khoảng) trên sân nhà của chính nó không, và mô hình đa năng có thua kém nhiều khi so sánh ở đâu không?